In [1]:
import sklearn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize
from collections import Counter
import re

In [2]:
COLS = ['sectionName', 'webTitle', 'bodyContent', 'webPublicationDate']
chunks = []
for chunk in pd.read_csv("guardian_articles.csv", usecols=COLS, chunksize=50000):
    chunks.append(chunk)
df = pd.concat(chunks, ignore_index=True)
print(f"Raw rows: {len(df)}")
 
# Expanded section mapping — groups related Guardian sections into 6 broad topics
# so every cluster has ample data and the cross-section bleed is interesting
SECTION_MAP = {
    # News / politics
    'World news':   'World News',
    'US news':      'World News',
    'UK news':      'UK/Politics',
    'Politics':     'UK/Politics',
    'News':         'UK/Politics',
    # Business / tech
    'Business':     'Business/Tech',
    'Technology':   'Business/Tech',
    # Environment / science
    'Environment':  'Env/Science',
    'Science':      'Env/Science',
    # Sport
    'Sport':        'Sport',
    'Football':     'Sport',
    # Culture / arts
    'Culture':      'Culture',
    'Music':        'Culture',
    'Film':         'Culture',
    'Books':        'Culture',
    'Art and design': 'Culture',
    'Stage':        'Culture',
}
 
sections_to_keep = list(SECTION_MAP.keys())
df = df[df['sectionName'].isin(sections_to_keep)].copy()
df['broadSection'] = df['sectionName'].map(SECTION_MAP)
print(f"After section filter: {len(df)}")
print(df['sectionName'].value_counts().to_string())

Raw rows: 149839
After section filter: 97328
sectionName
World news        15501
Football          10896
Sport             10376
US news            7480
Politics           7074
Business           6990
UK news            6481
Music              5760
Books              5121
Film               4990
Environment        4883
Stage              2520
Technology         2344
Art and design     2187
Culture            1774
Science            1623
News               1328


In [3]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'<[^>]+>', ' ', text)          # strip HTML tags
    text = re.sub(r'http\S+', ' ', text)           # strip URLs
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)      # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()
 
# Combine headline + body for richer features; fall back gracefully
text_col = 'bodyContent'
headline_col = 'webTitle'
 
df['combined_text'] = (df[headline_col].fillna('') + ' ' + df[text_col].fillna('')).apply(clean_text)

JUNK_TERMS = ['letters theguardian', 'guardian letters', 'email guardian',
              'join debate', 'debate email', 'letters click']
 
def is_junk(text):
    hits = sum(1 for t in JUNK_TERMS if t in text)
    return hits >= 2  # two or more junk signals → discard
 
before = len(df)
df = df[~df['combined_text'].apply(is_junk)].reset_index(drop=True)
print(f"After junk filter: {len(df)} (removed {before - len(df)} boilerplate articles)")
 
# Drop very short articles (likely empty after cleaning)
df = df[df['combined_text'].str.split().str.len() >= 20].reset_index(drop=True)
print(f"After text length filter: {len(df)}")
 
# Sample for tractability — 5000 gives good cluster resolution without OOM
if len(df) > 5000:
    total = len(df)
    sampled = []
    for section, grp in df.groupby('broadSection'):
        n = min(len(grp), int(5000 * len(grp) / total) + 1)
        sampled.append(grp.sample(n, random_state=42))
    df = (pd.concat(sampled, ignore_index=True)
            .sample(frac=1, random_state=42)
            .head(5000)
            .reset_index(drop=True))
    print(f"Stratified sample: {len(df)} articles")

After junk filter: 95879 (removed 1449 boilerplate articles)
After text length filter: 95313
Stratified sample: 5000 articles


In [4]:
# Features: TF-IDF weights on unigrams + bigrams, ignoring rare/common terms
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.85,
    stop_words='english'
)
X = vectorizer.fit_transform(df['combined_text'])
print(f"TF-IDF matrix: {X.shape}")
 
# Cosine similarity via L2-normalised vectors (standard for text)
X_norm = normalize(X, norm='l2')

TF-IDF matrix: (5000, 5000)


In [5]:
k_range = range(3, 12)
inertias, sil_scores = [], []
 
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = km.fit_predict(X_norm)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_norm, labels, metric='euclidean', sample_size=500, random_state=42))
    print(f"  k={k}  inertia={km.inertia_:.1f}  silhouette={sil_scores[-1]:.4f}")
 
best_k = k_range[np.argmax(sil_scores)]
print(f"\nBest k by silhouette: {best_k}")

  k=3  inertia=4703.4  silhouette=0.0118
  k=4  inertia=4676.7  silhouette=0.0112
  k=5  inertia=4651.4  silhouette=0.0129
  k=6  inertia=4633.6  silhouette=0.0108
  k=7  inertia=4616.8  silhouette=0.0121
  k=8  inertia=4605.9  silhouette=0.0121
  k=9  inertia=4591.3  silhouette=0.0121
  k=10  inertia=4576.9  silhouette=0.0117
  k=11  inertia=4568.2  silhouette=0.0131

Best k by silhouette: 11


In [7]:
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10, max_iter=500)
df['cluster'] = km_final.fit_predict(X_norm)
 

In [8]:
feature_names = vectorizer.get_feature_names_out()
 
def top_terms(cluster_id, n=15):
    center = km_final.cluster_centers_[cluster_id]
    idxs = center.argsort()[::-1][:n]
    return [(feature_names[i], round(center[i], 4)) for i in idxs]
 
print("\n── Cluster Top Terms ──")
for c in range(best_k):
    terms = [t for t, _ in top_terms(c)]
    section_dist = df[df['cluster'] == c]['sectionName'].value_counts().to_dict()
    print(f"\nCluster {c} (n={sum(df['cluster']==c)}): {', '.join(terms[:10])}")
    print(f"  Sections: {section_dist}")


── Cluster Top Terms ──

Cluster 0 (n=212): covid, health, said, cases, coronavirus, virus, people, pandemic, government, vaccine
  Sections: {'World news': 152, 'US news': 16, 'Politics': 11, 'UK news': 8, 'Science': 7, 'Sport': 7, 'Business': 3, 'News': 2, 'Technology': 1, 'Art and design': 1, 'Environment': 1, 'Books': 1, 'Culture': 1, 'Football': 1}

Cluster 1 (n=328): england, game, players, cricket, cup, team, ball, match, rugby, world
  Sections: {'Sport': 276, 'Football': 49, 'Science': 1, 'UK news': 1, 'Books': 1}

Cluster 2 (n=222): film, movie, films, cinema, like, star, story, movies, director, comedy
  Sections: {'Film': 194, 'Culture': 11, 'Stage': 7, 'Art and design': 3, 'News': 2, 'Science': 2, 'Music': 2, 'Books': 1}

Cluster 3 (n=540): said, company, bn, uk, government, year, bank, companies, workers, tax
  Sections: {'Business': 297, 'Technology': 61, 'World news': 46, 'UK news': 43, 'Politics': 25, 'Environment': 23, 'US news': 21, 'Sport': 6, 'Science': 4, 'News':

In [9]:
PALETTE = ['#E63946','#457B9D','#2A9D8F','#E9C46A','#F4A261',
           '#6A0572','#1D3557','#A8DADC','#F1FAEE','#264653']
 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Selecting k: Elbow & Silhouette", fontsize=14, fontweight='bold')
 
ax1 = axes[0]
ax1.plot(list(k_range), inertias, 'o-', color='#E63946', linewidth=2)
ax1.set_xlabel("Number of clusters (k)")
ax1.set_ylabel("Inertia")
ax1.set_title("Elbow Curve")
ax1.grid(alpha=0.3)
 
ax2 = axes[1]
ax2.plot(list(k_range), sil_scores, 's-', color='#457B9D', linewidth=2)
ax2.axvline(x=best_k, color='#E63946', linestyle='--', label=f'Best k={best_k}')
ax2.set_xlabel("Number of clusters (k)")
ax2.set_ylabel("Silhouette Score")
ax2.set_title("Silhouette Score")
ax2.legend()
ax2.grid(alpha=0.3)
 
plt.tight_layout()
plt.savefig("fig1_elbow_silhouette.png", dpi=150, bbox_inches='tight')
plt.close()
print("Saved fig1_elbow_silhouette.png")

Saved fig1_elbow_silhouette.png


In [10]:
svd = TruncatedSVD(n_components=50, random_state=42)
X_svd = svd.fit_transform(X_norm)
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_svd)
 
fig, ax = plt.subplots(figsize=(10, 7))
for c in range(best_k):
    mask = df['cluster'] == c
    top_t = top_terms(c, 3)
    label = f"C{c}: {', '.join([t for t, _ in top_t])}"
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               color=PALETTE[c % len(PALETTE)], alpha=0.45, s=15, label=label)
ax.set_title(f"Article Clusters in 2D PCA Space (k={best_k})", fontsize=13, fontweight='bold')
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.legend(loc='upper right', fontsize=8, framealpha=0.9)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig("fig2_cluster_scatter.png", dpi=150, bbox_inches='tight')
plt.close()
print("Saved fig2_cluster_scatter.png")

Saved fig2_cluster_scatter.png


In [11]:
ncols = 3
nrows = int(np.ceil(best_k / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3.5))
axes = axes.flatten()
 
for c in range(best_k):
    terms_scores = top_terms(c, 10)
    terms_list = [t for t, _ in terms_scores]
    scores = [s for _, s in terms_scores]
    ax = axes[c]
    bars = ax.barh(terms_list[::-1], scores[::-1], color=PALETTE[c % len(PALETTE)], alpha=0.85)
    ax.set_title(f"Cluster {c} (n={sum(df['cluster']==c)})", fontweight='bold')
    ax.set_xlabel("TF-IDF centroid weight")
    ax.grid(axis='x', alpha=0.3)
 
for c in range(best_k, len(axes)):
    axes[c].set_visible(False)
 
plt.suptitle("Top Terms per Cluster", fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("fig3_top_terms.png", dpi=150, bbox_inches='tight')
plt.close()
print("Saved fig3_top_terms.png")

Saved fig3_top_terms.png


In [12]:
# Figure 4: Section composition heatmap
df['broadSection'] = df['sectionName'].map(SECTION_MAP)
cross = pd.crosstab(df['cluster'], df['broadSection'])
cross_norm = cross.div(cross.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(cross_norm.values, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(cross_norm.columns)))
ax.set_xticklabels(cross_norm.columns, rotation=40, ha='right')
ax.set_yticks(range(len(cross_norm.index)))
ax.set_yticklabels([f"Cluster {i}" for i in cross_norm.index])
plt.colorbar(im, ax=ax, label='Proportion of cluster articles')
ax.set_title("Broad Section Composition of Each Cluster", fontsize=13, fontweight='bold')
for i in range(cross_norm.shape[0]):
    for j in range(cross_norm.shape[1]):
        val = cross_norm.values[i, j]
        ax.text(j, i, f"{val:.2f}", ha='center', va='center',
                fontsize=8, color='black' if val < 0.6 else 'white')
plt.tight_layout()
plt.savefig("fig4_section_heatmap.png", dpi=150, bbox_inches='tight')
plt.close()
print("Saved fig4_section_heatmap.png")

Saved fig4_section_heatmap.png


In [13]:
sizes = df['cluster'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(sizes.index, sizes.values,
       color=[PALETTE[i % len(PALETTE)] for i in sizes.index])
ax.set_xlabel("Cluster"); ax.set_ylabel("Number of Articles")
ax.set_title("Cluster Size Distribution", fontweight='bold')
ax.set_xticks(sizes.index)
ax.set_xticklabels([f"C{i}" for i in sizes.index])
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("fig5_cluster_sizes.png", dpi=150, bbox_inches='tight')
plt.close()
print("Saved fig5_cluster_sizes.png")

Saved fig5_cluster_sizes.png


In [14]:
print("\n── Sample Articles per Cluster ──")
rows = []
for c in range(best_k):
    for _, row in df[df['cluster'] == c].head(3).iterrows():
        rows.append({
            'Cluster': c,
            'BroadSection': row['broadSection'],
            'Section': row['sectionName'],
            'Title': str(row['webTitle'])[:100]
        })
examples_df = pd.DataFrame(rows)
print(examples_df.to_string(index=False))
examples_df.to_csv("cluster_examples.csv", index=False)
print("Saved cluster_examples.csv")


── Sample Articles per Cluster ──
 Cluster  BroadSection        Section                                                                                            Title
       0    World News     World news                             Minister rejects Dominic Cummings’ claims of ‘needless’ Covid deaths
       0    World News     World news                    Ardern urges New Zealanders to 'act like you have Covid-19' as lockdown looms
       0 Business/Tech     Technology                                                   The five: robots helping to tackle coronavirus
       1         Sport          Sport                      In Wales there will always be a welcome for referee Gaüzère | Robert Kitson
       1         Sport          Sport                                    Alapati Leiua try gives Bristol last laugh against rusty Bath
       1         Sport          Sport                  Toulouse European triumph throws up more questions than answers | Robert Kitson
       2       Cultu

In [15]:
final_sil = silhouette_score(
    X_norm, df['cluster'], metric='euclidean', sample_size=500, random_state=42
)
print(f"\nFinal silhouette score: {final_sil:.4f}")
print(f"Cluster sizes:\n{sizes.to_string()}")
print("\n✅ All outputs saved")


Final silhouette score: 0.0131
Cluster sizes:
cluster
0      212
1      328
2      222
3      540
4      371
5      209
6      531
7      349
8     1017
9      333
10     888

✅ All outputs saved
